# 2. Build the Transition Matrices

We split the **training** users into two groups and learn one Markov model for each group:

| Group | Label | Model |
|---|---|---|
| Will not call | `called = 0` | first-action distribution + transition matrix |
| Will call | `called = 1` | first-action distribution + transition matrix |

The model of a group answers the question:
*"If a user belongs to this group, how likely is it to see this sequence of actions?"*

**Output:** `models/markov_models.npz`.

## 2.1 Settings

`ALPHA` and `BETA` are smoothing constants. They are explained in step 2.4.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATA_DIR = Path("data")
MODELS_DIR = Path("models")

ALPHA = 1.0  # smoothing for transitions
BETA = 1.0   # smoothing for the first action

GROUP_NAMES = {0: "will NOT call", 1: "WILL call"}

## 2.2 Load the training data

Each user's actions are stored as text. We turn them into a list of numbers (action indices).

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")
true_process = np.load(DATA_DIR / "true_process.npz")

ACTIONS = [str(a) for a in true_process["actions"]]
N_ACTIONS = len(ACTIONS)
ACTION_INDEX = {action: i for i, action in enumerate(ACTIONS)}


def to_indices(actions_text):
    return [ACTION_INDEX[action] for action in actions_text.split()]


train["sequence"] = train["actions"].map(to_indices)
train[["user_id", "actions", "called"]].head()

## 2.3 Split the users into two groups

In [ ]:
sequences_by_group = {
    group: train.loc[train["called"] == group, "sequence"].tolist()
    for group in (0, 1)
}

for group, sequences in sequences_by_group.items():
    n_transitions = sum(len(s) - 1 for s in sequences)
    print(f"{GROUP_NAMES[group]:>14}: {len(sequences):>6} users, {n_transitions:>7} transitions")

Note that the callers group is much smaller. Its matrix is estimated from fewer transitions,
so it will be noisier.

## 2.4 Estimate a Markov model

We count two things in each group:

- $N_{\text{first}}(i)$ — how many users start with action $i$;
- $N(i \to j)$ — how many times action $j$ comes right after action $i$.

Then we turn the counts into probabilities:

$$\pi(i) = \frac{N_{\text{first}}(i) + \beta}{\sum_k N_{\text{first}}(k) + K\beta}
\qquad\qquad
A(i, j) = \frac{N(i \to j) + \alpha}{\sum_k N(i \to k) + K\alpha}$$

where $K$ is the number of actions.

**Why add $\alpha$ and $\beta$ (Laplace smoothing)?** Without it, a transition that never appeared in
training gets probability 0. Then any new user with that transition gets likelihood 0 — the model
says "impossible" only because it saw too little data. Adding a small count to every cell avoids this.
It matters most for the small callers group.

In [ ]:
def count_first_actions(sequences):
    counts = np.zeros(N_ACTIONS)
    for sequence in sequences:
        counts[sequence[0]] += 1
    return counts


def count_transitions(sequences):
    counts = np.zeros((N_ACTIONS, N_ACTIONS))
    for sequence in sequences:
        for current, following in zip(sequence[:-1], sequence[1:]):
            counts[current, following] += 1
    return counts


def estimate_markov_model(sequences, alpha=ALPHA, beta=BETA):
    first_counts = count_first_actions(sequences)
    transition_counts = count_transitions(sequences)
    first_action = (first_counts + beta) / (first_counts.sum() + N_ACTIONS * beta)
    matrix = (transition_counts + alpha) / (
        transition_counts.sum(axis=1, keepdims=True) + N_ACTIONS * alpha
    )
    return first_action, matrix, transition_counts


first_no_call, matrix_no_call, counts_no_call = estimate_markov_model(sequences_by_group[0])
first_call, matrix_call, counts_call = estimate_markov_model(sequences_by_group[1])

for matrix in (matrix_no_call, matrix_call):
    assert np.allclose(matrix.sum(axis=1), 1)

print("Transition counts in the WILL call group:")
pd.DataFrame(counts_call.astype(int), index=ACTIONS, columns=ACTIONS)

## 2.5 The two learned transition matrices

In [ ]:
def plot_matrix(ax, matrix, title, cmap="Blues", vmin=0, vmax=1):
    image = ax.imshow(matrix, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title)
    ax.set_xticks(range(N_ACTIONS))
    ax.set_xticklabels(ACTIONS, rotation=90)
    ax.set_yticks(range(N_ACTIONS))
    ax.set_yticklabels(ACTIONS)
    ax.set_xlabel("next action")
    ax.set_ylabel("current action")
    return image


fig, axes = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)
plot_matrix(axes[0], matrix_no_call, "Learned matrix: will NOT call")
image = plot_matrix(axes[1], matrix_call, "Learned matrix: WILL call")
fig.colorbar(image, ax=axes, shrink=0.7)
plt.show()

## 2.6 Which transitions point to a call?

For each transition we compute the **log ratio**:

$$\log \frac{A_{\text{call}}(i, j)}{A_{\text{no call}}(i, j)}$$

- positive (red) — this transition is more typical for users who call;
- negative (blue) — more typical for users who do not call;
- around zero — the transition does not help to separate the groups.

In notebook 3 a user is assigned to the group whose model gives the higher log-likelihood.
Every red transition the user makes raises the log-likelihood of the call group more than that of
the no-call group, so it pushes the user towards the call group.

In [ ]:
log_ratio = np.log(matrix_call) - np.log(matrix_no_call)
limit = np.abs(log_ratio).max()

fig, ax = plt.subplots(figsize=(8, 7), constrained_layout=True)
image = plot_matrix(ax, log_ratio, "Log ratio: log A_call − log A_no_call",
                    cmap="RdBu_r", vmin=-limit, vmax=limit)
fig.colorbar(image, ax=ax, shrink=0.7)
plt.show()

top = pd.DataFrame([
    {
        "transition": f"{ACTIONS[i]} → {ACTIONS[j]}",
        "P(no call model)": matrix_no_call[i, j],
        "P(call model)": matrix_call[i, j],
        "log ratio": log_ratio[i, j],
        "count in call group": int(counts_call[i, j]),
    }
    for i in range(N_ACTIONS) for j in range(N_ACTIONS)
]).sort_values("log ratio", ascending=False)

top.head(10).round(3)

## 2.7 Compare with the true matrices

We know the true matrices because we created the data. The error should be small for the large
no-call group and larger for the small callers group.

In [ ]:
errors = pd.DataFrame({
    "users": [len(sequences_by_group[0]), len(sequences_by_group[1])],
    "mean abs error (transitions)": [
        np.abs(matrix_no_call - true_process["matrix_no_call"]).mean(),
        np.abs(matrix_call - true_process["matrix_call"]).mean(),
    ],
    "mean abs error (first action)": [
        np.abs(first_no_call - true_process["first_action"]).mean(),
        np.abs(first_call - true_process["first_action"]).mean(),
    ],
}, index=[GROUP_NAMES[0], GROUP_NAMES[1]])
errors.round(4)

## 2.8 The prior: how common is calling?

The share of callers in the training data is the **prior probability** $P(\text{call})$.
The training users were not filtered, so this share is a fair estimate of the real call rate.
Notebook 3 uses it to compare two decision rules.

In [ ]:
call_rate = train["called"].mean()
print(f"P(call) estimated from training data: {call_rate:.4f}")

## 2.9 Save the model

In [ ]:
MODELS_DIR.mkdir(exist_ok=True)
np.savez(
    MODELS_DIR / "markov_models.npz",
    actions=np.array(ACTIONS),
    first_no_call=first_no_call,
    matrix_no_call=matrix_no_call,
    first_call=first_call,
    matrix_call=matrix_call,
    call_rate=call_rate,
    alpha=ALPHA,
    beta=BETA,
)
print(f"Saved {MODELS_DIR / 'markov_models.npz'}")